In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [2]:
pip install xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import xgboost as xgb
import lightgbm as lgb
import joblib

In [15]:
df = pd.read_csv("Datasets/Final_Features/final_bowler_dataset.csv")

In [31]:
#Separate Features & Target
TARGET = "target_next_match_wickets"

X = df.drop(columns=[TARGET])
y = df[TARGET]

In [32]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print("Train Size:", X_train.shape)
print("Test Size :", X_test.shape)

Train Size: (10636, 18)
Test Size : (2660, 18)


In [34]:
# BASELINE MODEL

baseline_pred = X_test["form_wickets_last_10"]

baseline_pred = np.clip(baseline_pred, 0, None)

print("MAE :", mean_absolute_error(y_test, baseline_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, baseline_pred)))
print("R2  :", r2_score(y_test, baseline_pred))

MAE : 0.8880794247523572
RMSE: 1.135737030758579
R2  : -0.10496112948909686


In [43]:
# RANDOM FOREST MODEL

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_pred = np.clip(rf_pred, 0, None)

print("RF MAE:", mean_absolute_error(y_test, rf_pred))
print("RF RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("RF R2:", r2_score(y_test, rf_pred))

RF MAE: 0.8325192897421327
RF RMSE: 1.0556015234891207
RF R2: 0.045465906532478506


In [44]:
# XGBOOST MODEL

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.06,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_pred = np.clip(xgb_pred, 0, None)

print("XGB MAE:", mean_absolute_error(y_test, xgb_pred))
print("XGB RMSE:", np.sqrt(mean_squared_error(y_test, xgb_pred)))
print("XGB R2:", r2_score(y_test, xgb_pred))

XGB MAE: 0.8563882664644292
XGB RMSE: 1.0810248260887552
XGB R2: -0.0010661208302302772


In [45]:
# LIGHTGBM MODEL

lgb_model = lgb.LGBMRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X_train, y_train)

lgb_pred = lgb_model.predict(X_test)
lgb_pred = np.clip(lgb_pred, 0, None)

print("LGB MAE:", mean_absolute_error(y_test, lgb_pred))
print("LGB RMSE:", np.sqrt(mean_squared_error(y_test, lgb_pred)))
print("LGB R2:", r2_score(y_test, lgb_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000610 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2005
[LightGBM] [Info] Number of data points in the train set: 10636, number of used features: 18
[LightGBM] [Info] Start training from score 1.001598
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

In [46]:
# HYPERPARAMETER TUNING

# RANDOM FOREST 

from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf = RandomForestRegressor(random_state=42)

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

rf_pred = best_rf.predict(X_test)
rf_pred = np.clip(rf_pred, 0, None)

print("Best RF Params:", grid_rf.best_params_)
print("Tuned RF MAE:", mean_absolute_error(y_test, rf_pred))
print("Tuned RF RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("Tuned RF R2:", r2_score(y_test, rf_pred))

Fitting 3 folds for each of 162 candidates, totalling 486 fits
Best RF Params: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 600}
Tuned RF MAE: 0.8295097084031978
Tuned RF RMSE: 1.05548181481273
Tuned RF R2: 0.04568238885329012


In [47]:
# XGBOOST RANDOMIZED SEARCH 

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42
)

param_dist = {
    "n_estimators": [300, 500, 800],
    "max_depth": [4, 5, 6, 7],
    "learning_rate": [0.03, 0.05, 0.07],
    "subsample": [0.75, 0.8, 0.85],
    "colsample_bytree": [0.75, 0.8, 0.85]
}

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=2,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_xgb = random_search.best_estimator_

xgb_pred = best_xgb.predict(X_test)
xgb_pred = np.clip(xgb_pred, 0, None)

print("BEST PARAMETERS:", random_search.best_params_)
print("BEST XGB MAE:", mean_absolute_error(y_test, xgb_pred))
print("BEST XGB RMSE:", np.sqrt(mean_squared_error(y_test, xgb_pred)))
print("BEST XGB R2:", r2_score(y_test, xgb_pred))

Fitting 3 folds for each of 25 candidates, totalling 75 fits
BEST PARAMETERS: {'subsample': 0.75, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.75}
BEST XGB MAE: 0.8362101944317495
BEST XGB RMSE: 1.0594032703957756
BEST XGB R2: 0.0385780199783623


In [48]:
joblib.dump(best_xgb, "bowler_rfmodel.joblib")

['bowler_rfmodel.joblib']